# Librerías 

In [ ]:
from Extraction.full_extraction import extraer_data_completa
from to_labelbox import enviar_a_labelbox

In [1]:
import pandas as pd

In [ ]:
%pip install scikit-learn
%pip install spacy

In [ ]:
import spacy

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Dataset

In [ ]:
df =extraer_data_completa(None)

# EDA

Observamos el shape de nuestro dataframe, la presencia de nulos y duplicados y sí están estandarizados los datos.

In [3]:
df

,Unnamed: 0,Titulo,URL,Cuerpo,Autor,Medio,Pais,Fecha
0,0,"Dane señala que pobreza en Colombia bajó, pero...",https://www.semana.com/economia/macroeconomia/...,El Departamento Administrativo Nacional de Est...,Manuel Santiago Sánchez González,Semana,Colombia,"Sep 15, 2026"
1,1,Exfiscal venezolano advierte sobre expansión d...,https://www.semana.com/nacion/medellin/articul...,Las autoridades anunciaron un duro golpe al Tr...,Juan Diego Valencia Martínez,Semana,Colombia,"Sep 20, 2026"
2,2,Vuelven los vuelos de deportación a Venezuela:...,https://www.semana.com/mundo/articulo/vuelven-...,Estados Unidos reanudó este lunes los vuelos d...,Juan Felipe Useche Chacón,Semana,Colombia,"Aug 04, 2026"
3,3,Venezolanos se unen por Colombia tras terremot...,https://www.semana.com/nacion/articulo/venezol...,La solidaridad de los venezolanos por el terre...,Gabriel Salazar López,Semana,Colombia,"Aug 12, 2026"
4,4,Denuncian detención de un periodista nicaragüe...,https://www.semana.com/mundo/articulo/denuncia...,"El periodista nicaragüense Luis Galeano, direc...",Juan David Cardozo Maglioni,Semana,Colombia,"Sep 14, 2026"
...,...,...,...,...,...,...,...,...
156,156,EU extiende amparo migratorio para migrantes v...,https://www.milenio.com/internacional/estados-...,El gobierno del presidente estadunidense Joe B...,AFP,Milenio,México,2022-07-11 20:06
157,157,Sale caravana de migrantes venezolanos desde T...,https://www.milenio.com/estados/caravana-migra...,"Una caravana de migrantes, en su mayoría venez...",Julio Navarro Cárdenas,Milenio,México,2022-06-24 13:56
158,158,"Entre calor y fatiga, migrantes quedan varados...",https://www.milenio.com/politica/migrantes-olv...,Suscríbete a Milenio y descarga la edición imp...,Redacción,Milenio,México,2022-06-17 15:37
159,159,"""Plan Vuelta a la Patria"": Maduro promete trip...",https://www.milenio.com/internacional/plan-vue...,"El presidente de Venezuela, Nicolás Maduro, pr...",Editorial Milenio,Milenio,México,2022-02-02 19:47


In [4]:
df.shape

(161, 8)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 161 entries, 0 to 160
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  161 non-null    int64
 1   Titulo      161 non-null    str  
 2   URL         161 non-null    str  
 3   Cuerpo      160 non-null    str  
 4   Autor       161 non-null    str  
 5   Medio       161 non-null    str  
 6   Pais        161 non-null    str  
 7   Fecha       161 non-null    str  
dtypes: int64(1), str(7)
memory usage: 10.2 KB


In [17]:
duplicates = df[df.duplicated()]
print(duplicates)

Empty DataFrame
Columns: [ID, Titulo, URL, Cuerpo, Autor, Medio, Pais, Fecha]
Index: []


# Preprocesamiento

## Limpieza e ingeniería de datos

#### Erradicación filas con cuerpo nulo

In [6]:
df = df.dropna(how='any')

#### Adición de la columna ID 

In [7]:

# Eliminar la primera columna (la columna de ID actual sin título)
df = df.iloc[:, 1:]

# Crear la nueva columna ID
df.insert(
    0,  # posición (primera columna)
    "ID",
    [f"articulo_{i}" for i in range(1, len(df) + 1)]
)

In [8]:
df

,ID,Titulo,URL,Cuerpo,Autor,Medio,Pais,Fecha
0,articulo_1,"Dane señala que pobreza en Colombia bajó, pero...",https://www.semana.com/economia/macroeconomia/...,El Departamento Administrativo Nacional de Est...,Manuel Santiago Sánchez González,Semana,Colombia,"Sep 15, 2026"
1,articulo_2,Exfiscal venezolano advierte sobre expansión d...,https://www.semana.com/nacion/medellin/articul...,Las autoridades anunciaron un duro golpe al Tr...,Juan Diego Valencia Martínez,Semana,Colombia,"Sep 20, 2026"
2,articulo_3,Vuelven los vuelos de deportación a Venezuela:...,https://www.semana.com/mundo/articulo/vuelven-...,Estados Unidos reanudó este lunes los vuelos d...,Juan Felipe Useche Chacón,Semana,Colombia,"Aug 04, 2026"
3,articulo_4,Venezolanos se unen por Colombia tras terremot...,https://www.semana.com/nacion/articulo/venezol...,La solidaridad de los venezolanos por el terre...,Gabriel Salazar López,Semana,Colombia,"Aug 12, 2026"
4,articulo_5,Denuncian detención de un periodista nicaragüe...,https://www.semana.com/mundo/articulo/denuncia...,"El periodista nicaragüense Luis Galeano, direc...",Juan David Cardozo Maglioni,Semana,Colombia,"Sep 14, 2026"
...,...,...,...,...,...,...,...,...
156,articulo_156,EU extiende amparo migratorio para migrantes v...,https://www.milenio.com/internacional/estados-...,El gobierno del presidente estadunidense Joe B...,AFP,Milenio,México,2022-07-11 20:06
157,articulo_157,Sale caravana de migrantes venezolanos desde T...,https://www.milenio.com/estados/caravana-migra...,"Una caravana de migrantes, en su mayoría venez...",Julio Navarro Cárdenas,Milenio,México,2022-06-24 13:56
158,articulo_158,"Entre calor y fatiga, migrantes quedan varados...",https://www.milenio.com/politica/migrantes-olv...,Suscríbete a Milenio y descarga la edición imp...,Redacción,Milenio,México,2022-06-17 15:37
159,articulo_159,"""Plan Vuelta a la Patria"": Maduro promete trip...",https://www.milenio.com/internacional/plan-vue...,"El presidente de Venezuela, Nicolás Maduro, pr...",Editorial Milenio,Milenio,México,2022-02-02 19:47


Debemos estandarizar las fechas y anonimizar a los autores:

#### Estandarización de las fechas 

In [9]:
df["Fecha"] = pd.to_datetime(
    df["Fecha"],
    format="mixed",   # útil para formatos distintos
    errors="coerce"
)

df["Fecha"] = df["Fecha"].dt.strftime("%Y-%m-%d")

In [10]:
df

,ID,Titulo,URL,Cuerpo,Autor,Medio,Pais,Fecha
0,articulo_1,"Dane señala que pobreza en Colombia bajó, pero...",https://www.semana.com/economia/macroeconomia/...,El Departamento Administrativo Nacional de Est...,Manuel Santiago Sánchez González,Semana,Colombia,2026-09-15
1,articulo_2,Exfiscal venezolano advierte sobre expansión d...,https://www.semana.com/nacion/medellin/articul...,Las autoridades anunciaron un duro golpe al Tr...,Juan Diego Valencia Martínez,Semana,Colombia,2026-09-20
2,articulo_3,Vuelven los vuelos de deportación a Venezuela:...,https://www.semana.com/mundo/articulo/vuelven-...,Estados Unidos reanudó este lunes los vuelos d...,Juan Felipe Useche Chacón,Semana,Colombia,2026-08-04
3,articulo_4,Venezolanos se unen por Colombia tras terremot...,https://www.semana.com/nacion/articulo/venezol...,La solidaridad de los venezolanos por el terre...,Gabriel Salazar López,Semana,Colombia,2026-08-12
4,articulo_5,Denuncian detención de un periodista nicaragüe...,https://www.semana.com/mundo/articulo/denuncia...,"El periodista nicaragüense Luis Galeano, direc...",Juan David Cardozo Maglioni,Semana,Colombia,2026-09-14
...,...,...,...,...,...,...,...,...
156,articulo_156,EU extiende amparo migratorio para migrantes v...,https://www.milenio.com/internacional/estados-...,El gobierno del presidente estadunidense Joe B...,AFP,Milenio,México,2022-07-11
157,articulo_157,Sale caravana de migrantes venezolanos desde T...,https://www.milenio.com/estados/caravana-migra...,"Una caravana de migrantes, en su mayoría venez...",Julio Navarro Cárdenas,Milenio,México,2022-06-24
158,articulo_158,"Entre calor y fatiga, migrantes quedan varados...",https://www.milenio.com/politica/migrantes-olv...,Suscríbete a Milenio y descarga la edición imp...,Redacción,Milenio,México,2022-06-17
159,articulo_159,"""Plan Vuelta a la Patria"": Maduro promete trip...",https://www.milenio.com/internacional/plan-vue...,"El presidente de Venezuela, Nicolás Maduro, pr...",Editorial Milenio,Milenio,México,2022-02-02


#### Anonimización de los autores

In [11]:
autores_unicos = df["Autor"].unique()

In [12]:
df_autores = pd.DataFrame({
    "ID_Autor": [f"AutorID_{i}" for i in range(1, len(autores_unicos) + 1)],
    "Autor": autores_unicos
}
)

In [13]:
df_autores.to_csv(
    "Data/dataset_autores_anonimizados.csv",
    index = False,
    encoding= "utf-8-sig"
)

df_autores


,ID_Autor,Autor
0,AutorID_1,Manuel Santiago Sánchez González
1,AutorID_2,Juan Diego Valencia Martínez
2,AutorID_3,Juan Felipe Useche Chacón
3,AutorID_4,Gabriel Salazar López
4,AutorID_5,Juan David Cardozo Maglioni
...,...,...
88,AutorID_89,César Velázquez
89,AutorID_90,Milenio Digital
90,AutorID_91,Editorial Milenio
91,AutorID_92,Jaime Zambrano


In [14]:
# Crear diccionario Autor -> AutorID
mapa_autores = dict(zip(df_autores["Autor"], df_autores["ID_Autor"]))

# Reemplazar nombres por IDs
df["Autor"] = df["Autor"].map(mapa_autores)

In [15]:
df

,ID,Titulo,URL,Cuerpo,Autor,Medio,Pais,Fecha
0,articulo_1,"Dane señala que pobreza en Colombia bajó, pero...",https://www.semana.com/economia/macroeconomia/...,El Departamento Administrativo Nacional de Est...,AutorID_1,Semana,Colombia,2026-09-15
1,articulo_2,Exfiscal venezolano advierte sobre expansión d...,https://www.semana.com/nacion/medellin/articul...,Las autoridades anunciaron un duro golpe al Tr...,AutorID_2,Semana,Colombia,2026-09-20
2,articulo_3,Vuelven los vuelos de deportación a Venezuela:...,https://www.semana.com/mundo/articulo/vuelven-...,Estados Unidos reanudó este lunes los vuelos d...,AutorID_3,Semana,Colombia,2026-08-04
3,articulo_4,Venezolanos se unen por Colombia tras terremot...,https://www.semana.com/nacion/articulo/venezol...,La solidaridad de los venezolanos por el terre...,AutorID_4,Semana,Colombia,2026-08-12
4,articulo_5,Denuncian detención de un periodista nicaragüe...,https://www.semana.com/mundo/articulo/denuncia...,"El periodista nicaragüense Luis Galeano, direc...",AutorID_5,Semana,Colombia,2026-09-14
...,...,...,...,...,...,...,...,...
156,articulo_156,EU extiende amparo migratorio para migrantes v...,https://www.milenio.com/internacional/estados-...,El gobierno del presidente estadunidense Joe B...,AutorID_72,Milenio,México,2022-07-11
157,articulo_157,Sale caravana de migrantes venezolanos desde T...,https://www.milenio.com/estados/caravana-migra...,"Una caravana de migrantes, en su mayoría venez...",AutorID_84,Milenio,México,2022-06-24
158,articulo_158,"Entre calor y fatiga, migrantes quedan varados...",https://www.milenio.com/politica/migrantes-olv...,Suscríbete a Milenio y descarga la edición imp...,AutorID_64,Milenio,México,2022-06-17
159,articulo_159,"""Plan Vuelta a la Patria"": Maduro promete trip...",https://www.milenio.com/internacional/plan-vue...,"El presidente de Venezuela, Nicolás Maduro, pr...",AutorID_91,Milenio,México,2022-02-02


#### Etiquetado

Usamos el servicio labelbox para añadir rigurosidad al etiquetado:

In [ ]:
enviar_a_labelbox(df)

# Procesamiento

In [ ]:
!python -m spacy download es_core_news_sm

In [ ]:
# Cargar modelo en español
nlp = spacy.load('es_core_news_sm')